# Crafting Effective CLAUDE.md Files for AI Agent Projects

CLAUDE.md is the configuration file that Claude Code reads at the start of every session to understand your project. It tells the agent what commands to run, how the codebase is structured, what conventions to follow, and what rules to never break. A well-written CLAUDE.md is the difference between an agent that works reliably on your project and one that constantly makes mistakes, uses the wrong tools, or ignores your team's conventions.

This notebook walks through the full lifecycle of a CLAUDE.md: generating one from real project files, spotting the anti-patterns that break agents, reviewing and scoring it with Claude, and running programmatic validation that catches wrong-ecosystem commands and stale file paths before they reach CI.


## Prerequisites

Before running this notebook, ensure you have:

- **Python 3.11+** and the dependencies installed (run the cell below)
- **Anthropic API key** set in a `.env` file as `ANTHROPIC_API_KEY=sk-ant-...`
  or exported in your shell environment

Get your API key at [console.anthropic.com](https://console.anthropic.com/settings/keys).

In [1]:
%%capture
%pip install -U anthropic python-dotenv

In [2]:
import json
import re
import tempfile
from pathlib import Path

import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

## Part 1 — The Anatomy of an Effective CLAUDE.md

In [3]:
ANATOMY_TEMPLATE = """
# CLAUDE.md — Project Configuration for AI Agents

## Project Overview
# PURPOSE: Gives the agent immediate context about what this project does,
# the tech stack, and what domain knowledge it needs. Without this, the agent
# has to infer context from file names — which is slow and error-prone.
<project name and one-line description>
<primary language and framework>
<key domain or business context>

## Build & Test Commands
# PURPOSE: The single most critical section. An agent that runs the wrong
# command (e.g., `npm test` when the project uses `pnpm test`) will fail
# silently or produce misleading errors. Every command here must be verified.
- Install dependencies: <exact command>
- Start dev server:     <exact command>
- Run all tests:        <exact command>
- Run a single test:    <exact command with file/pattern arg>
- Build for prod:       <exact command>
- Lint:                 <exact command>
- Type check:           <exact command>

## Architecture Overview
# PURPOSE: Helps the agent navigate the codebase without reading every file.
# Describes which directories contain which concerns, and where to look for
# specific functionality. Prevents the agent from creating files in wrong places.
<key directories and what they contain>
<entry points and how requests flow through the system>
<database or storage layer>

## Code Style & Conventions
# PURPOSE: Prevents the agent from introducing style inconsistencies that
# will fail linting or irritate reviewers. The agent cannot infer these
# reliably from reading code — explicit rules are always more reliable.
<indentation: tabs or spaces, width>
<quote style: single or double>
<naming conventions: camelCase, snake_case, PascalCase per context>
<import ordering rules>
<max line length>

## Key Rules & Constraints
# PURPOSE: Hard boundaries the agent must never cross, regardless of what
# seems convenient. These prevent irreversible mistakes (deleting data,
# pushing secrets, breaking prod). Phrase as absolute prohibitions.
- NEVER commit .env files or hardcode API keys
- NEVER modify the database schema directly — use migrations
- NEVER push directly to main — always use a PR
- NEVER run destructive commands (DROP TABLE, rm -rf) without confirmation
"""

print(ANATOMY_TEMPLATE)


# CLAUDE.md — Project Configuration for AI Agents

## Project Overview
# PURPOSE: Gives the agent immediate context about what this project does,
# the tech stack, and what domain knowledge it needs. Without this, the agent
# has to infer context from file names — which is slow and error-prone.
<project name and one-line description>
<primary language and framework>
<key domain or business context>

## Build & Test Commands
# PURPOSE: The single most critical section. An agent that runs the wrong
# command (e.g., `npm test` when the project uses `pnpm test`) will fail
# silently or produce misleading errors. Every command here must be verified.
- Install dependencies: <exact command>
- Start dev server:     <exact command>
- Run all tests:        <exact command>
- Run a single test:    <exact command with file/pattern arg>
- Build for prod:       <exact command>
- Lint:                 <exact command>
- Type check:           <exact command>

## Architecture Overview
# PURPOSE: Helps t

## Part 2 — Generating a CLAUDE.md from Project Files

Generating a CLAUDE.md upfront works well for new projects. For existing projects with established conventions, derive rules from agent failures instead — every rule should exist because the agent got it wrong without it. Start with a generated skeleton, then replace generic advice with battle-tested rules from real sessions.

In [4]:
SAMPLE_PROJECT = {
    "package.json": """{
  "name": "pipeline-api",
  "version": "1.0.0",
  "scripts": {
    "dev": "tsx watch src/index.ts",
    "build": "tsc -p tsconfig.json",
    "test": "vitest run",
    "test:watch": "vitest",
    "lint": "eslint src --ext .ts",
    "typecheck": "tsc --noEmit"
  },
  "dependencies": {
    "fastify": "^4.0.0",
    "mongoose": "^7.0.0",
    "zod": "^3.0.0"
  },
  "devDependencies": {
    "typescript": "^5.0.0",
    "tsx": "^4.0.0",
    "vitest": "^1.0.0",
    "eslint": "^8.0.0"
  }
}""",
    "src/index.ts": """import Fastify from 'fastify'
import { pipelineRoutes } from './routes/pipelines'
import { authPlugin } from './plugins/auth'

const app = Fastify({ logger: true })
app.register(authPlugin)
app.register(pipelineRoutes, { prefix: '/api/v1' })
app.listen({ port: 3000 })""",
    "src/routes/pipelines.ts": """import { FastifyPluginAsync } from 'fastify'
import { PipelineService } from '../services/transform'

export const pipelineRoutes: FastifyPluginAsync = async (app) => {
  app.get('/pipelines', async (req, reply) => {
    return PipelineService.list()
  })
  app.post('/pipelines', async (req, reply) => {
    return PipelineService.create(req.body)
  })
}""",
    "src/plugins/auth.ts": """import fp from 'fastify-plugin'
import { FastifyPluginAsync } from 'fastify'

export const authPlugin: FastifyPluginAsync = fp(async (app) => {
  app.addHook('onRequest', async (req, reply) => {
    const token = req.headers.authorization?.split(' ')[1]
    if (!token) reply.code(401).send({ error: 'Unauthorized' })
  })
})""",
    "src/services/transform.ts": """import { Pipeline } from '../models/pipeline'

export const PipelineService = {
  list: () => Pipeline.find(),
  create: (data: unknown) => Pipeline.create(data),
  delete: (id: string) => Pipeline.findByIdAndDelete(id),
}""",
}

project_context = "\n\n".join(
    f"=== {filename} ===\n{content}" for filename, content in SAMPLE_PROJECT.items()
)

response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1500,
    system=(
        "You are an expert CLAUDE.md writer. "
        "Generate a complete, accurate CLAUDE.md for the project files provided. "
        "Only use information found in the actual files — never invent commands, "
        "paths, or conventions that are not supported by the files. "
        "Keep the output under 150 lines."
    ),
    messages=[
        {
            "role": "user",
            "content": f"Generate a CLAUDE.md for this project:\n\n{project_context}",
        }
    ],
)

generated_claude_md = response.content[0].text
print(generated_claude_md)

# CLAUDE.md

## Project Overview
`pipeline-api` is a Fastify-based REST API with MongoDB (via Mongoose) and Zod validation, written in TypeScript.

## Commands

```bash
# Development
npm run dev          # Run with hot reload (tsx watch)

# Build
npm run build        # Compile TypeScript

# Testing
npm test             # Run tests once (vitest)
npm run test:watch   # Run tests in watch mode

# Code Quality
npm run lint         # ESLint on src/ (.ts files)
npm run typecheck    # Type-check without emitting
```

## Architecture

```
src/
├── index.ts              # Fastify app entry point (port 3000)
├── plugins/
│   └── auth.ts           # Auth plugin — validates Bearer token on every request
├── routes/
│   └── pipelines.ts      # Pipeline route handlers, mounted at /api/v1
├── services/
│   └── transform.ts      # PipelineService — list, create, delete
└── models/
    └── pipeline.ts       # Mongoose model (Pipeline)
```

## API
All routes are prefixed `/api/v1` and require an `Author

## Part 3 — Anti-Patterns That Break Agents

In [5]:
ANTI_PATTERNS = [
    {
        "name": "Contradictory Instructions",
        "bad": "## Dependencies\n- Install: `pip install -r requirements.txt`\n\n"
        "## Dev Setup\n- Create venv: `uv venv && uv sync`",
        "good": "## Dependencies\n- Install: `uv sync`\n"
        "- Add a package: `uv add <package>`\n"
        "- Never use pip directly in this project",
        "why": "The #1 failure mode in long-lived CLAUDE.md files. As rules accumulate "
        "across sessions, contradictions creep in — 'use uv' in one section, "
        "'pip install' in another. Unlike wrong-ecosystem errors that fail "
        "loudly, contradictions fail silently: the agent picks one rule, "
        "ignores the other, and you don't notice until something breaks "
        "downstream. Deduplicate rules ruthlessly after every edit.",
    },
    {
        "name": "Wrong Ecosystem Commands",
        "bad": "- Build: `cargo build`\n- Test: `cargo test`",
        "good": "- Build: `npm run build`\n- Test: `npm test`",
        "why": "Commands from the wrong ecosystem fail immediately. An agent in a "
        "Node.js project that tries `cargo build` will waste time diagnosing "
        "a 'command not found' error instead of doing useful work.",
    },
    {
        "name": "Stale File Paths",
        "bad": "- Entry point: `src/server.js`\n- Config: `config/app.json`",
        "good": "- Entry point: `src/index.ts`\n- Config: `src/config/settings.ts`",
        "why": "File paths go stale when code is refactored but CLAUDE.md is not updated. "
        "An agent sent to modify `src/server.js` will create a new file instead "
        "of editing the correct one, silently diverging from the real codebase.",
    },
    {
        "name": "Missing Environment Variables",
        "bad": "- Start the server: `npm run dev`",
        "good": "- Copy `.env.example` to `.env` and fill in values before starting\n"
        "- Required env vars: DATABASE_URL, JWT_SECRET, PORT\n"
        "- Start the server: `npm run dev`",
        "why": "The server crashes on startup if required env vars are missing. "
        "An agent following just the start command will see a cryptic error "
        "and waste several turns trying to diagnose a missing env var.",
    },
    {
        "name": "Excessive Length",
        "bad": "500+ lines of background context, project history, team philosophy, "
        "and aspirational goals before the key commands appear on line 400.",
        "good": "Commands in the first 20 lines, architecture in the next 30. "
        "Total under 100 lines. Links to external docs for background.",
        "why": "Critical commands buried deep in a long file get missed. Agents "
        "process CLAUDE.md like a system prompt — front-loaded information "
        "gets more attention. A 500-line CLAUDE.md is a liability.",
    },
    {
        "name": "Vague Test Commands",
        "bad": "- Run tests: `npm test`",
        "good": "- Run all tests: `npm test`\n"
        "- Run a single test file: `npx vitest run src/routes/pipelines.test.ts`\n"
        "- Run tests matching pattern: `npx vitest run -t 'should create pipeline'`",
        "why": "An agent fixing one bug shouldn't have to run the entire test suite "
        "to verify its change. Without a single-test command, it either runs "
        "everything (slow, expensive) or skips verification entirely (risky).",
    },
    {
        "name": "Stale Dynamic State",
        "bad": "## Current Sprint\n- Finish auth migration (started March 3)\n"
        "- Deploy v2.1 to staging by Friday",
        "good": "## Dynamic Context\n"
        "- Check `docs/current-sprint.md` at session start for active priorities\n"
        "- Run `./scripts/session-init.sh` to load current deploy status",
        "why": "Static rules (indentation, test commands) age well. Dynamic state "
        "(sprint goals, deploy status, active priorities) goes stale within days. "
        "Hardcoding it into CLAUDE.md guarantees the agent acts on outdated context. "
        "Point to a live source or use a session-init hook instead.",
    },
]

for i, pattern in enumerate(ANTI_PATTERNS, 1):
    print(f"{'=' * 60}")
    print(f"Anti-Pattern {i}: {pattern['name']}")
    print(f"{'=' * 60}")
    print(f"\n❌ BAD:\n{pattern['bad']}")
    print(f"\n✅ GOOD:\n{pattern['good']}")
    print(f"\n⚠️  WHY: {pattern['why']}")
    print()

Anti-Pattern 1: Wrong Ecosystem Commands

❌ BAD:
- Build: `cargo build`
- Test: `cargo test`

✅ GOOD:
- Build: `npm run build`
- Test: `npm test`

⚠️  WHY: Commands from the wrong ecosystem fail immediately. An agent in a Node.js project that tries `cargo build` will waste time diagnosing a 'command not found' error instead of doing useful work.

Anti-Pattern 2: Stale File Paths

❌ BAD:
- Entry point: `src/server.js`
- Config: `config/app.json`

✅ GOOD:
- Entry point: `src/index.ts`
- Config: `src/config/settings.ts`

⚠️  WHY: File paths go stale when code is refactored but CLAUDE.md is not updated. An agent sent to modify `src/server.js` will create a new file instead of editing the correct one, silently diverging from the real codebase.

Anti-Pattern 3: Contradictory Instructions

❌ BAD:
- Use tabs for indentation
- Use 2-space indentation throughout

✅ GOOD:
- Use 2-space indentation (spaces only, no tabs)

⚠️  WHY: Contradictory rules force the agent to guess, and it will guess inc

## Part 4 — Using Claude to Review Your CLAUDE.md

In [6]:
PROBLEMATIC_CLAUDE_MD = """
# My Project

This is a great project we've been building for two years.

## Commands
- Build: `cargo build --release`
- Test: `cargo test`
- Start: `npm run dev`

## Code Style
- Use tabs for indentation
- Use 2 spaces for indentation
- camelCase for variables

## Files
- Main entry: `src/server.js`
- Database config: `config/database.yml`

## Notes
Just run npm start and it should work.
"""

ACTUAL_PROJECT_FILES = """
The project is a Node.js/TypeScript REST API.
package.json scripts: dev, build (tsc), test (vitest), lint (eslint)
Main entry point: src/index.ts (not src/server.js)
Database: MongoDB via Mongoose (no database.yml)
Required env vars: DATABASE_URL, JWT_SECRET
.eslintrc uses 2-space indentation with spaces (no tabs)
"""

review_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1200,
    system=(
        "You are a CLAUDE.md reviewer. Given a CLAUDE.md and a description of the "
        "actual project, identify every issue in the CLAUDE.md. "
        "Be direct and specific. Format your response as a numbered list where "
        "each item describes: the problem, its impact on an AI agent, and the fix."
    ),
    messages=[
        {
            "role": "user",
            "content": (
                f"CLAUDE.md to review:\n{PROBLEMATIC_CLAUDE_MD}\n\n"
                f"Actual project reality:\n{ACTUAL_PROJECT_FILES}"
            ),
        }
    ],
)

print(review_response.content[0].text)

Here are all the issues in the CLAUDE.md:

1. **`cargo build --release` and `cargo test` are wrong language/toolchain.** The project is Node.js/TypeScript, not Rust. An AI agent running these commands will get "command not found" or similar errors and waste time debugging a nonexistent Rust setup. Fix: Replace with `npm run build` (runs `tsc`) and `npm run vitest` or `npm test`.

2. **`npm run dev` is listed as "Start" but there is no explanation of required env vars.** `npm run dev` will likely crash immediately without `DATABASE_URL` and `JWT_SECRET` set. An AI agent will see a runtime failure with no clear cause. Fix: Document required environment variables and how to set them (e.g., `.env` file or export commands) before running the server.

3. **Contradictory indentation rules: "Use tabs" and "Use 2 spaces" are both listed.** An AI agent cannot follow both rules simultaneously and will make an arbitrary choice, likely producing code that fails linting. Fix: Remove the tabs rule; k

## Part 5 — Programmatic Validation

In [7]:
def detect_project_ecosystem(project_dir: str) -> dict:
    """Detect build ecosystem from config files.

    Returns dict with keys: type, config_file, expected_commands.
    Checks in order: package.json, Cargo.toml, go.mod,
    pyproject.toml, requirements.txt, pom.xml, build.gradle.
    """
    checks = [
        ("package.json", "node", ["npm", "npx", "pnpm", "yarn", "bun"]),
        ("Cargo.toml", "rust", ["cargo"]),
        ("go.mod", "go", ["go"]),
        ("pyproject.toml", "python", ["python", "pip", "uv", "poetry", "pytest"]),
        ("requirements.txt", "python", ["python", "pip", "pytest"]),
        ("pom.xml", "java", ["mvn", "java"]),
        ("build.gradle", "java", ["gradle", "java"]),
    ]
    for config_file, ecosystem_type, expected_commands in checks:
        if Path(project_dir, config_file).exists():
            return {
                "type": ecosystem_type,
                "config_file": config_file,
                "expected_commands": expected_commands,
            }
    return {"type": "unknown", "config_file": None, "expected_commands": []}


def validate_claude_md_commands(claude_md_content: str, ecosystem: dict) -> list[dict]:
    """Find commands in backticks that belong to wrong ecosystem.

    Returns list of {severity, command, issue, fix}.
    """
    issues = []
    wrong_ecosystem_commands = {
        "node": ["cargo", "go run", "go build", "python", "pip", "mvn", "gradle"],
        "rust": ["npm", "npx", "pip", "python", "go run", "mvn"],
        "go": ["npm", "npx", "cargo", "pip", "python", "mvn"],
        "python": ["npm", "npx", "cargo", "go run", "go build", "mvn"],
        "java": ["npm", "npx", "cargo", "go run", "pip", "python"],
    }
    backtick_pattern = re.compile(r"`([^`]+)`")
    found_commands = backtick_pattern.findall(claude_md_content)
    forbidden = wrong_ecosystem_commands.get(ecosystem["type"], [])
    for cmd in found_commands:
        cmd_lower = cmd.strip().lower()
        for bad_prefix in forbidden:
            if cmd_lower.startswith(bad_prefix):
                issues.append(
                    {
                        "severity": "error",
                        "command": cmd,
                        "issue": f"Wrong ecosystem command for {ecosystem['type']} project",
                        "fix": f"Replace with the equivalent {ecosystem['type']} command",
                    }
                )
                break
    return issues


def validate_file_references(claude_md_content: str, project_dir: str) -> list[dict]:
    """Find file paths in backticks that do not exist on disk.

    Matches extensions: ts, js, py, rs, go, json, yaml, yml, toml, md.
    Returns list of {severity, file, issue, fix}.
    """
    issues = []
    path_pattern = re.compile(r"`([^`]*\.(?:ts|js|py|rs|go|json|yaml|yml|toml|md))`")
    referenced_files = path_pattern.findall(claude_md_content)
    for file_path in referenced_files:
        full_path = Path(project_dir) / file_path.lstrip("/")
        if not full_path.exists():
            issues.append(
                {
                    "severity": "warning",
                    "file": file_path,
                    "issue": f"File referenced in CLAUDE.md does not exist: {file_path}",
                    "fix": "Update the path or remove the reference",
                }
            )
    return issues


# Demo: create a temporary Node.js project with intentional problems
with tempfile.TemporaryDirectory() as tmp_dir:
    # Create a Node.js project structure
    Path(tmp_dir, "package.json").write_text('{"name": "demo", "scripts": {"test": "vitest"}}')
    Path(tmp_dir, "src").mkdir()
    Path(tmp_dir, "src", "index.ts").write_text("// entry point")

    # Write a CLAUDE.md with cargo commands (wrong ecosystem) and stale path
    bad_claude_md = """
## Commands
- Build: `cargo build --release`
- Test: `cargo test`
- Entry point: `src/server.js`
"""

    ecosystem = detect_project_ecosystem(tmp_dir)
    print(f"Detected ecosystem: {ecosystem['type']} (via {ecosystem['config_file']})")
    print(f"Expected commands:  {ecosystem['expected_commands']}\n")

    cmd_issues = validate_claude_md_commands(bad_claude_md, ecosystem)
    file_issues = validate_file_references(bad_claude_md, tmp_dir)

    all_issues = cmd_issues + file_issues
    if all_issues:
        print(f"Found {len(all_issues)} issue(s):\n")
        for issue in all_issues:
            if "command" in issue:
                print(f"  ❌ [{issue['severity'].upper()}] Command: `{issue['command']}`")
                print(f"     Issue: {issue['issue']}")
                print(f"     Fix:   {issue['fix']}")
            else:
                print(f"  ❌ [{issue['severity'].upper()}] File: `{issue['file']}`")
                print(f"     Issue: {issue['issue']}")
                print(f"     Fix:   {issue['fix']}")
            print()
    else:
        print("✅ No issues found")

Detected ecosystem: node (via package.json)
Expected commands:  ['npm', 'npx', 'pnpm', 'yarn', 'bun']

Found 3 issue(s):

  ❌ [ERROR] Command: `cargo build --release`
     Issue: Wrong ecosystem command for node project
     Fix:   Replace with the equivalent node command

  ❌ [ERROR] Command: `cargo test`
     Issue: Wrong ecosystem command for node project
     Fix:   Replace with the equivalent node command

  ❌ [WARNING] File: `src/server.js`
     Issue: File referenced in CLAUDE.md does not exist: src/server.js
     Fix:   Update the path or remove the reference



## Part 6 — Subdirectory CLAUDE.md for Monorepos

In [8]:
MONOREPO_STRUCTURE = """
Monorepo structure:

/ (root)
  packages/
    api/          — Node.js/Fastify REST API, uses npm, vitest, TypeScript
    frontend/     — Next.js 14 app, uses pnpm, Jest, React, Tailwind
    ml-service/   — Python FastAPI service, uses uv, pytest, PyTorch
  infra/
    terraform/    — AWS infrastructure, uses terraform, tfvars files
  .github/
    workflows/    — CI/CD pipelines for each package

Each package has its own package manager, test runner, and build toolchain.
The root has a Makefile that delegates to each package.
"""

monorepo_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=2000,
    system=(
        "You are an expert in monorepo tooling and AI agent configuration. "
        "Given a monorepo structure, explain the CLAUDE.md strategy for it."
    ),
    messages=[
        {
            "role": "user",
            "content": (
                f"Given this monorepo structure:\n{MONOREPO_STRUCTURE}\n\n"
                "Answer these questions:\n"
                "1. What should go in the root CLAUDE.md?\n"
                "2. Which subdirectories need their own CLAUDE.md, and why?\n"
                "3. What specific content should each subdirectory CLAUDE.md contain?\n"
                "4. What must NOT be duplicated between root and subdirectory files?"
            ),
        }
    ],
)

print(monorepo_response.content[0].text)

# CLAUDE.md Strategy for This Monorepo

## 1. What Goes in the Root CLAUDE.md

The root file establishes **monorepo-wide context and navigation**. It answers: "what is this repo, how does it fit together, and how do I operate it as a whole?"

```markdown
# Root CLAUDE.md

## Repository Overview
Full-stack application with REST API, Next.js frontend, ML inference service,
and AWS infrastructure. Each package is independently deployable.

## Repository Map
- packages/api/       — Node.js/Fastify REST API (npm, vitest, TypeScript)
- packages/frontend/  — Next.js 14 frontend (pnpm, Jest, React, Tailwind)
- packages/ml-service/— Python FastAPI ML service (uv, pytest, PyTorch)
- infra/terraform/    — AWS infrastructure (Terraform)

## Cross-Package Commands (via Makefile)
make test-all         # Run all test suites
make build-all        # Build all packages
make dev              # Start all services locally
make lint-all         # Lint everything

See each package's CLAUDE.md for package-spe

## Part 7 — Quality Scoring

**Limitation:** This scorer evaluates document structure — completeness, consistency, conciseness. It cannot measure whether the agent actually follows the rules at runtime. A CLAUDE.md can score 10/10 here and still produce bad agent behavior if rules are technically correct but practically ambiguous. The ultimate validation is running the agent and checking its output against your rules.

In [9]:
def score_claude_md(content: str) -> dict:
    """Score a CLAUDE.md on five quality dimensions using Claude.

    Returns dict with scores (1-10), overall, top_improvements, summary.
    Handles JSONDecodeError gracefully.
    """
    schema = """
    {
      "scores": {
        "completeness": <1-10>,
        "accuracy": <1-10>,
        "conciseness": <1-10>,
        "actionability": <1-10>,
        "consistency": <1-10>
      },
      "overall": <1-10>,
      "top_improvements": ["...", "...", "..."],
      "summary": "one sentence"
    }
    """
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=600,
        system=(
            "You are a CLAUDE.md quality evaluator. Score the provided CLAUDE.md "
            "on five dimensions (1-10 each): "
            "completeness (are all key sections present?), "
            "accuracy (do commands and paths look correct?), "
            "conciseness (is it free of unnecessary padding?), "
            "actionability (can an agent act on it immediately?), "
            "consistency (are there any contradictions?). "
            f"Respond ONLY with valid JSON matching this schema: {schema}"
        ),
        messages=[{"role": "user", "content": f"Score this CLAUDE.md:\n\n{content}"}],
    )
    raw = response.content[0].text.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = re.sub(r"^```[^\n]*\n", "", raw)
        raw = re.sub(r"\n```$", "", raw.strip())
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {
            "scores": {
                "completeness": 0,
                "accuracy": 0,
                "conciseness": 0,
                "actionability": 0,
                "consistency": 0,
            },
            "overall": 0,
            "top_improvements": ["Could not parse response — check model output"],
            "summary": "Scoring failed due to JSON parse error.",
        }


result = score_claude_md(generated_claude_md)

# Visual bar chart
BAR_FULL = "█"
BAR_EMPTY = "░"
BAR_WIDTH = 10

print("CLAUDE.md Quality Scores")
print("=" * 40)
for dimension, score in result["scores"].items():
    filled = round(score)
    bar = BAR_FULL * filled + BAR_EMPTY * (BAR_WIDTH - filled)
    print(f"  {dimension:<16} [{bar}] {score}/10")

print()
print(f"  Overall score:   {result['overall']}/10")
print(f"  Summary:         {result['summary']}")
print()
print("Top 3 improvements:")
for i, improvement in enumerate(result["top_improvements"][:3], 1):
    print(f"  {i}. {improvement}")

CLAUDE.md Quality Scores
  completeness     [███████░░░] 7/10
  accuracy         [████████░░] 8/10
  conciseness      [█████████░] 9/10
  actionability    [████████░░] 8/10
  consistency      [████████░░] 8/10

  Overall score:   8/10
  Summary:         A clean, well-structured CLAUDE.md that covers commands, architecture, and conventions concisely, but lacks environment variable documentation and has a minor inconsistency between the API table and the described service capabilities.

Top 3 improvements:
  1. Add environment variable documentation (e.g., MongoDB connection string, Bearer token secret, PORT) so an agent can configure and run the app without guessing
  2. Expand the API table to include DELETE and any other endpoints mentioned in the service (list/create/delete are all listed in PipelineService but only GET/POST appear in the table)
  3. Clarify testing setup — mention test runner config file location, how to set up test DB or mocks, and whether integration vs unit tests

## Part 8 — Before and After

In [10]:
BEFORE = """
# My Awesome Project

This is the backend service we've been working on. It does a lot of cool stuff
with data pipelines and was started in 2022.

## How to run things
- Build: `cargo build`
- Test: `npm test` (or sometimes `python -m pytest`, check with the team)
- Start: just run main
- Lint: we use tabs and 4 spaces

## Important files
- Main: `src/server.js`
- Config: `config.yml` (might be config.json now, not sure)

## Rules
- Be careful with the database
- Don't break things
"""

AFTER = """
# pipeline-api — Node.js/TypeScript REST API for data pipeline management

## Build & Test Commands
- Install dependencies:  `npm install`
- Start dev server:      `npm run dev`
- Run all tests:         `npm test`
- Run a single test:     `npx vitest run src/routes/pipelines.test.ts`
- Build for prod:        `npm run build`
- Lint:                  `npm run lint`
- Type check:            `npm run typecheck`

## Environment Setup
- Copy `.env.example` to `.env` before starting
- Required: DATABASE_URL, JWT_SECRET, PORT

## Architecture
- Entry point:    `src/index.ts` — Fastify app setup and plugin registration
- Routes:         `src/routes/` — one file per resource (pipelines, jobs)
- Services:       `src/services/` — business logic, no HTTP concerns
- Plugins:        `src/plugins/` — auth, rate limiting, database connection
- Database:       MongoDB via Mongoose (`src/models/`)

## Code Style
- Language:   TypeScript strict mode
- Indent:     2 spaces (no tabs)
- Quotes:     single quotes
- Naming:     camelCase for variables/functions, PascalCase for types/classes
- Max length: 100 characters

## Key Rules
- NEVER commit `.env` files or hardcode secrets
- NEVER modify MongoDB collections directly — use Mongoose models
- NEVER push directly to `main` — open a PR and request review
- ALWAYS run `npm run typecheck` before committing
"""

separator = "=" * 60
print(f"{'BEFORE — Broken CLAUDE.md':^60}")
print(separator)
print(BEFORE)
print()
print(f"{'AFTER — Clean CLAUDE.md':^60}")
print(separator)
print(AFTER)

                 BEFORE — Broken CLAUDE.md                  

# My Awesome Project

This is the backend service we've been working on. It does a lot of cool stuff
with data pipelines and was started in 2022.

## How to run things
- Build: `cargo build`
- Test: `npm test` (or sometimes `python -m pytest`, check with the team)
- Start: just run main
- Lint: we use tabs and 4 spaces

## Important files
- Main: `src/server.js`
- Config: `config.yml` (might be config.json now, not sure)

## Rules
- Be careful with the database
- Don't break things


                  AFTER — Clean CLAUDE.md                   

# pipeline-api — Node.js/TypeScript REST API for data pipeline management

## Build & Test Commands
- Install dependencies:  `npm install`
- Start dev server:      `npm run dev`
- Run all tests:         `npm test`
- Run a single test:     `npx vitest run src/routes/pipelines.test.ts`
- Build for prod:        `npm run build`
- Lint:                  `npm run lint`
- Type check:        

## Summary

| Principle | Rule |
|---|---|
| **Accuracy** | Every command must be verified against the actual project — never guess |
| **Ecosystem match** | Commands must belong to the project's actual build system |
| **File paths** | All referenced files must exist at the stated paths |
| **Conciseness** | Keep it under 100 lines; commands in the first 20 |
| **No contradictions** | One rule per topic — never specify tabs AND spaces. Audit for contradictions after every edit; they are the #1 silent failure mode |
| **Static vs. dynamic** | Hardcode static rules (style, commands). Point dynamic state (sprint, deploy status) to a live source or session-init hook |
| **Env vars** | Document all required environment variables before the start command |
| **Freshness** | Update CLAUDE.md in the same PR as any rename, move, or command change |
| **Monorepos** | Root file covers shared tooling; subdirectory files cover per-package commands |

## Next Steps

- **Automate validation in CI** — wire the ecosystem detector and command validator from Part 5 into a CI step so every PR is checked
- **Add a pre-commit hook** — run the file-reference validator locally before each commit to catch stale paths immediately
- **Keep CLAUDE.md in the same PR as the code change** — treat it like a migration: the rename, the path update, and the CLAUDE.md fix are one atomic commit